In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, date_format, regexp_extract

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA silver")

In [0]:
df = spark.table("bronze.cpih")

df_components = spark.table("config.cpih_components")
valid_codes = [row["code"] for row in df_components.collect()]
print(f"Filtering to codes: {valid_codes}")

In [0]:

from pyspark.sql.functions import udf
from pyspark.sql.types import DateType
from datetime import datetime

# Spark's to_date with "MMM-yy" format misreads two-digit years as 2097 etc.
# Parse manually using Python's datetime, forcing the correct century
def parse_mmm_yy(s):
    if s is None:
        return None
    try:
        dt = datetime.strptime(s, "%b-%y")
        # strptime treats 00-68 as 2000-2068, 69-99 as 1969-1999
        # which is correct for this dataset
        return dt.date()
    except:
        return None

parse_date_udf = udf(parse_mmm_yy, DateType())

df_silver = (
    df
    .filter(col("cpih1dim1aggid").isin(valid_codes))
    .filter(regexp_extract(col("mmm-yy"), r"^\w{3}-\d{2}$", 0) != "")
    .withColumnRenamed("v4_0", "index_value")
    .withColumnRenamed("cpih1dim1aggid", "component_code")
    .withColumnRenamed("Aggregate", "component_name")
    .withColumn("index_value", col("index_value").cast("double"))
    .withColumn("report_date", parse_date_udf(col("mmm-yy")))
    .withColumn("year_month", date_format(col("report_date").cast("timestamp"), "yyyy-MM"))
    .select("report_date", "year_month", "component_code", "component_name", "index_value")
)

In [0]:
df_silver.show(10)
df_silver.printSchema()
print(f"Row count: {df_silver.count()}")
print(f"Components: {[row["component_code"] for row in df_silver.select("component_code").distinct().collect()]}")


In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.cpih")
)

In [0]:
# silver.cpih
# Filtered to 4 components: CP00, CP01, CP04, CP07
# Monthly rows only, mmm-yy format
# report_date (date), year_month (string), component_code (string),
# component_name (string), index_value (double)